In [0]:
from typing import Any

In [0]:
import logging

from pyspark.sql.functions import col
from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_qc(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "ExposureId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        # "IngestionTimestamp timestamp, BatchId string, ExposureDebtorKey string, CaseCode int,"
        # "ExposureSourceSystemKey string, AgreementKey string, PartyKey string, ExposureKey string,"
        # "PartyTaxRegistrationNumber string, CDI string, PartyName string, PartyRelationshipTypeCode string,"
        # "PartyRelationshipType string, PrimaryPartyFlag boolean, AllBorrowersDeceasedFlag boolean,"
        # "ProductKey string, ProductCode int, ExposureTypeCode string, ExposureType string,"
        # "ExposureStatusCode string, ExposureStatus string, ExposureStatusdate date, Opendate date,"
        # "ActiveExpiryDate date, Maturitydate date, OriginBank string, OriginationAmount decimal(19,2),"
        # "Currency string, BankingUnitCode int, BankingUnit string, MechanismCode string, Mechanism string,"
        # "BranchCode string, Branch string, PurposeReasonCode int, PurposeReason string,"
        # "PurposeKTCode int, PurposeKT string, MaterialityStartDate timestamp,"
        # "DaysPastDueWithMateriality decimal(15,0), DaysPastDue int, DaysPastDuePrincipal int,"
        # "DaysPastDueinterest string, BucketCode string, Bucket string, FirstArreardate date,"
        # "ExpectedOverduedate date, Overduedate date, Defaultdate date,  Denounceddate date,"
        # "MasterExposureKey string, SlaveFlag boolean,"
        # "InstalmentFrequency int, InstallmentFixedAmount decimal(19,2), NumberOfInstallments int,"
        # "FirstInstallmentDate date, NextInstallmentdate date, NextInstallmentAmount decimal(19,2),"
        # "PreviousInstallmentdate date, LastBillingdate date, NumberOfUnpaidBills string,"
        # "LastRestructuringTypeCode int, LastRestructuringType string, LastRestructuringDate date, RestructuringPEEdate date,"
        # "RestructuringPEETypeCode int, RestructuringPEEType string, ObjectiveStatusCode string,"
        # "ObjectiveStatus string, ObjectiveStatusdate date, GracePeriodEnddate date,"
        # "GracePeriodTypeCode string, GracePeriodType string, GracePeriodinterestPayableFlag string,"
        # "GracePeriodNextinterestdate timestamp, GracePeriodinterestFrequency decimal(3,0),"
        # "OffBalance decimal(19,2), OnBalance decimal(19,2), OutstandingTotalBalance string,"
        # "Principal decimal(19,2), interest decimal(19,2), Unrealizedinterest decimal(19,2),"
        # "Commission  decimal(19,2), Expense decimal(19,2), Accrued decimal(19,2),"
        # "DueAmount decimal(26,4), DuePrincipal decimal (19,2), Dueinterest decimal(19,2),"
        # "DueUnrealizedinterest decimal(19,2), DueCommission decimal (19,2), DueExpense decimal(19,2),"
        # "DueAccrued decimal(19,2), Penaltyinterest decimal(19,2),PenaltyUnrealizedinterest decimal(19,2),"
        # "InterestRate decimal(19,6), InterestCode int, InterestDescription string,"
        # "BaseRate decimal(19,6), BaseCode int, BaseDescription string,"
        # "N128Rate decimal(19,6), SpreadRate decimal(19,6), PenaltyRate decimal(19,6),"
        # "InterestRateTypeCode string, InterestRateType string, SpreadCurrencyCode string,"
        # "SpreadCurrency string, SpreadPeriod string, InterestDuration int, DayCounts int,"
        # "InterestFrequency string, InterestFixedDate date, interestValiditydate date,"
        # "InterestChangeStartdate date, InterestChangeEnddate date, InterestRateUpdatedate date,"
        # "InterestCalcStopFlag string, AutoRenewalFlag string, IsActiveFlag boolean,WriteOffAmount decimal(19,4),"
        # "WriteOffDate date,AssetSegmentCode string, AssetSegment string, LPSCode string,"
        # "LPS string,LPSEng string, LPSDate string, DiasNumber string,"
        # "ProvisionFrozenInterestAmount string, AccountNticeBUKey string, SayFlag string,"
        # "InsuranceCompany string, PhysicalNumber string, PreCourtDate string, CourtDate string,"
        # "ContractSigningDate string, ExposureLimitAmount string, BaloonAmount decimal(15,2),"
        # "AdministrationStatusCode integer, AdministrationStatus integer, Tenor string,"
        # "IsSoldFlag boolean, IsSubsidyFlag string, IsGuaranteeFlag boolean, EverDenouncedFlag string,"
        # "EverDenouncedDate string, SPVCode string, SPV string, TotalOverdueInsterest decimal(4,0),"
        # "TotalOverdueInstallments decimal(4,0), OutstandingTotalBalanceMLT decimal(19,4),"
        # "TreatmentBranchCode int, TreatmentBranch string, LetterStatusCode string, LetterStatus string,"
        # "LetterStatusDate date, CoCStatusCode string, CoCStatus string, CoCAccountStatusDate date,"
        # "FirstLetterDate date, SecondLetterDate date, ThirdLetterDate date,"
        # "FirstLetterReminderDate date, FirstLetterFlag boolean, FirstLetterReminderFlag boolean,"
        # "SecondLetterFlag boolean, ThirdLetterFlag boolean, ExclusionDate date, ExclusionReasonCode string,"
        # "ExclusionReason string, LegalActionPriority string, DueDebtAmount decimal(19,4), BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Exposure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qc",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== case_master ==================

        case_master_extra_filters = None

        if case_master_extra_filters is not None:
            case_master_extra_filters.append(date_filter)
            case_master_final_filters = " AND ".join(case_master_extra_filters)
        else:
            case_master_final_filters = date_filter

        case_master_selected_fields = [
            "debt_code",
            "vend_code",
            "qc_bu",
            "old_case_code",
            "case_code",
            "case_code_primary",
            "contract_number",
            "cust_code",
            "LPS",
            "lps_status",
            "lps_date",
            "case_relation_type_code",
            "account_type",
            "case_currently_open_flag",
            "Origination_Date",
            "Case_Exp_Date",
            "branch_code",
            "denounce_status",
            "Denounce_Status_Date",
            "master_account",
            "objective status",
            "objective status date",
            "off_balance",
            "on_balance",
            "total_balance",
            "WriteOff_Date",
            "CEXT_GENERIC_BIT_03",
            "casp_generic_int_09_code",
            "CoC_AccountStatus",
            "CoC_AccountStatusDate",
            "First_CoC_LetterDate",
            "Second_CoC_LetterDate",
            "Third_CoC_LetterDate",
            "CEXT_GENERIC_DATETIME_08",
            "CEXT_GENERIC_BIT_01",
            "CEXT_GENERIC_BIT_04",
            "CEXT_GENERIC_BIT_02",
            "CEXT_GENERIC_BIT_03",
            "CEXT_DEDICATED_PROPERTY_01_DATETIME",
            "CEXT_DEDICATED_PROPERTY_02_NVARCHAR",
            "businessdate",
        ]

        case_master = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_case_master",
            case_master_final_filters,
            case_master_selected_fields,
        )

        case_master = case_master.withColumnRenamed(
            "objective status", "ObjectiveStatus"
        ).withColumnRenamed("objective status date", "ObjectiveStatusDate")

        case_master = case_master.filter(col("case_currently_open_flag") == "1")

        # ================== debt ==================

        debt_extra_filters = None

        if debt_extra_filters is not None:
            debt_extra_filters.append(date_filter)
            debt_final_filters = " AND ".join(debt_extra_filters)
        else:
            debt_final_filters = date_filter

        debt_selected_fields = [
            "debt_code",
            "currency",
            "dpd",
            "bucket_code",
            "SPRD_CODE",
            "installment_amount",
            "last_billing_date",
            "principal",
            "interest_amount",
            "expenses",
            "overdue_other_amount",
            "overdue_principal",
            "DEBT_ADMINISTRATION_STATUS_CODE",
        ]

        debt = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_debt",
            debt_final_filters,
            debt_selected_fields,
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "cdi",
            "tax_number",
            "vendor_code",
            "cust_code",
            "first_name",
            "last_name",
        ]

        customer = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ====== filter the cdi ======
        case_master_open_keys = (
            case_master
            .filter(col("case_currently_open_flag") == "1")
            .select(
                col("vend_code").alias("vendor_code"),
                col("cust_code")
            )
            .distinct()
        )

        customer = customer.join(
            case_master_open_keys,
            on=["vendor_code", "cust_code"],
            how="inner",
        )
        # ======  ======


        # =================== lst_of_val =================

        lst_of_val_extra_filters = None

        if lst_of_val_extra_filters is not None:
            lst_of_val_extra_filters.append(date_filter)
            lst_of_val_final_filters = " AND ".join(lst_of_val_extra_filters)
        else:
            lst_of_val_final_filters = date_filter

        lst_of_val_selected_fields = ["LOV_DESC", "LOVT_DESC", "LOV_CODE"]

        lst_of_val = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lst_of_val",
            lst_of_val_final_filters,
            lst_of_val_selected_fields,
        )

        # ================== lps_priority ==============

        lps_priority_extra_filters = None

        if lps_priority_extra_filters is not None:
            lps_priority_extra_filters.append(date_filter)
            lps_priority_final_filters = " AND ".join(lps_priority_extra_filters)
        else:
            lps_priority_final_filters = date_filter

        lps_priority_selected_fields = ["LPS_code", "LPS"]

        lps_priority = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qc_lps_priority",
            lps_priority_final_filters,
            lps_priority_selected_fields,
        )

        lps_priority = lps_priority.distinct()

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "vendor_code_qualco", "originBank"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Joins ==================

        exposure_qc = (
            case_master.join(
                debt, (case_master["debt_code"] == debt["debt_code"]), "left"
            )
            .join(
                customer,
                (case_master["vend_code"] == customer["vendor_code"])
                & (case_master["cust_code"] == customer["cust_code"]),
                "left",
            )
            .join(
                lps_priority,
                (lps_priority.LPS_code == case_master.LPS),
                "left",
            )
            .join(
                lst_of_val,
                (lst_of_val.LOVT_DESC == "RELATION_TYPES")
                & (case_master.case_relation_type_code == lst_of_val.LOV_CODE),
                "left",
            )
            .join(
                portfolio,
                (case_master["vend_code"] == portfolio["vendor_code_qualco"]),
                "left",
            )
        )

        exposure_qc_primary_key_fields = [
            case_master.old_case_code,
            case_master.cust_code,
            lst_of_val.LOV_DESC,
            lit("qc"),
        ]

        exposuredebtorkey_qc_primary_key_fields = [
            case_master.old_case_code,
            case_master.cust_code,
            lst_of_val.LOV_DESC,
        ]

        exposure_qc = (
            (
                exposure_qc.withColumn(
                    "ExposureId",
                    sha2(
                        concat_ws(";", *exposure_qc_primary_key_fields),
                        256,
                    ),
                )
            )
            .withColumn(
                "ExposureStatus",
                when(case_master.case_currently_open_flag == "0", "Closed").otherwise(
                    "Active"
                ),
            )
            .withColumn(
                "ExposureDebtorKey",
                concat_ws(";", *exposuredebtorkey_qc_primary_key_fields),
            )
            .withColumn(
                "PartyName",
                when(customer.first_name.isin(None), customer.last_name).otherwise(
                    concat_ws(" ", customer.first_name, customer.last_name)
                ),
            )
            .withColumn(
                "FirstLetterFlag",
                when(case_master.CEXT_GENERIC_BIT_01.isNull(), lit(False)).otherwise(
                    case_master.CEXT_GENERIC_BIT_01
                ),
            )
            .withColumn(
                "FirstLetterReminderFlag",
                when(case_master.CEXT_GENERIC_BIT_04.isNull(), lit(False)).otherwise(
                    case_master.CEXT_GENERIC_BIT_04
                ),
            )
            .withColumn(
                "SecondLetterFlag",
                when(case_master.CEXT_GENERIC_BIT_02.isNull(), lit(False)).otherwise(
                    case_master.CEXT_GENERIC_BIT_02
                ),
            )
            .withColumn(
                "ThirdLetterFlag",
                when(case_master.CEXT_GENERIC_BIT_03.isNull(), lit(False)).otherwise(
                    case_master.CEXT_GENERIC_BIT_03
                ),
            )
            .withColumn(
                "ExclusionDate",
                when(
                    case_master.CEXT_DEDICATED_PROPERTY_01_DATETIME.isNull(),
                    "1900-01-01",
                ).otherwise(case_master.CEXT_DEDICATED_PROPERTY_01_DATETIME),
            )
            .withColumn(
                "ExclusionReasonCode",
                when(
                    case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR.isNull()
                    | (case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR == ""),
                    lit(None),
                ).otherwise(case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR),
            )
            .withColumn(
                "ExclusionReason",
                when(
                    case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR.isNull()
                    | (case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR == ""),
                    lit(None),
                ).otherwise(case_master.CEXT_DEDICATED_PROPERTY_02_NVARCHAR),
            )
            .withColumn(
                "PrimaryPartyFlag",
                when(
                    case_master.case_code == case_master.case_code_primary, "True"
                ).otherwise("False"),
            )
        )

        # ================== Selection of Final Columns ==================

        exposure_qc = exposure_qc.select(
            exposure_qc.ExposureId.alias("ExposureId"),
            lit("qc").alias("SourceSystem"),
            exposure_qc.vend_code.cast("string").alias("PortfolioKey"),
            exposure_qc.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            exposure_qc.ExposureDebtorKey.cast("string").alias("ExposureDebtorKey"),
            exposure_qc.case_code.cast("int").alias("CaseCode"),
            debt.debt_code.cast("string").alias("ExposureSourceSystemKey"),
            case_master.old_case_code.cast("string").alias("ExposureKey"),
            exposure_qc.contract_number.cast("string").alias("AgreementKey"),
            case_master.cust_code.cast("string").alias("PartyKey"),
            customer.cdi.cast("string").alias("CDI"),
            exposure_qc.PartyName.cast("string").alias("PartyName"),
            customer.tax_number.cast("string").alias("PartyTaxRegistrationNumber"),
            exposure_qc.case_relation_type_code.cast("string").alias(
                "PartyRelationshipTypeCode"
            ),
            exposure_qc.LOV_DESC.cast("string").alias("PartyRelationshipType"),
            exposure_qc.PrimaryPartyFlag.cast("boolean").alias("PrimaryPartyFlag"),
            debt.SPRD_CODE.cast("string").alias("ProductKey"),
            debt.SPRD_CODE.alias("ProductCode"),
            exposure_qc.account_type.cast("string").alias("ExposureTypeCode"),
            lit(None).cast("string").alias("ExposureType"),
            exposure_qc.denounce_status.cast("string").alias("ExposureStatusCode"),
            exposure_qc.denounce_status.cast("string").alias("ExposureStatus"),
            lit(None).cast("date").alias("ExposureStatusDate"),
            exposure_qc.Origination_Date.cast("date").alias("OpenDate"),
            lit(None).alias("ActiveExpiryDate"),
            exposure_qc.Case_Exp_Date.cast("date").alias("MaturityDate"),
            portfolio.originBank.cast("string").alias("OriginBank"),
            lit(None).cast("decimal(19, 2)").alias("OriginationAmount"),
            exposure_qc.currency.cast("string").alias("Currency"),
            lit(None).cast("Int").alias("BankingUnitCode"),
            lit(None).cast("string").alias("BankingUnit"),
            lit(None).cast("string").alias("MechanismCode"),
            lit(None).cast("string").alias("Mechanism"),
            exposure_qc.branch_code.cast("string").alias("BranchCode"),
            exposure_qc.branch_code.cast("string").alias("Branch"),
            lit(None).cast("int").alias("PurposeReasonCode"),
            lit(None).cast("string").alias("PurposeReason"),
            lit(None).cast("int").alias("PurposeKTCode"),
            lit(None).cast("string").alias("PurposeKT"),
            lit(None).alias("MaterialityStartDate"),
            lit(None).alias("DaysPastDueWithMateriality"),
            exposure_qc.dpd.cast("int").alias("DaysPastDue"),
            lit(None).cast("int").alias("DaysPastDuePrincipal"),
            lit(None).cast("string").alias("DaysPastDueInterest"),
            exposure_qc.bucket_code.cast("string").alias("BucketCode"),
            exposure_qc.bucket_code.cast("string").alias("Bucket"),
            lit(None).cast("date").alias("FirstArrearDate"),
            lit(None).cast("date").alias("ExpectedOverdueDate"),
            lit(None).cast("date").alias("OverdueDate"),
            lit(None).cast("date").alias("DefaultDate"),
            # exposure_qc.denounce_status.cast("string").alias("DenouncedStatusCode"),
            # exposure_qc.denounce_status.cast("string").alias("DenouncedStatus"),
            exposure_qc.Denounce_Status_Date.cast("date").alias("DenouncedDate"),
            exposure_qc.master_account.alias("MasterExposureKey"),
            lit(None).cast("boolean").alias("SlaveFlag"),
            lit(None).cast("int").alias("InstalmentFrequency"),
            exposure_qc.installment_amount.cast("decimal(19,2)").alias(
                "InstallmentFixedAmount"
            ),
            lit(None).cast("int").alias("NumberOfInstallments"),
            lit(None).alias("FirstInstallmentDate"),
            lit(None).cast("date").alias("NextInstallmentDate"),
            lit(None).alias("NextInstallmentAmount"),
            lit(None).cast("date").alias("PreviousInstallmentDate"),
            exposure_qc.last_billing_date.cast("date").alias("LastBillingDate"),
            lit(None).cast("string").alias("NumberOfUnpaidBills"),
            lit(None).cast("int").alias("LastRestructuringTypeCode"),
            lit(None).cast("string").alias("LastRestructuringType"),
            lit(None).cast("date").alias("LastRestructuringDate"),
            lit(None).cast("date").alias("RestructuringPEEDate"),
            lit(None).alias("RestructuringPEETypeCode"),
            lit(None).alias("RestructuringPEEType"),
            exposure_qc.ObjectiveStatus.cast("string").alias("ObjectiveStatusCode"),
            # exposure_qc.ObjectiveStatusDate.cast("string").alias("ObjectiveStatus"),
            exposure_qc.ObjectiveStatus.cast("string").alias("ObjectiveStatus"), # REPBTC-90 KS 10/06/2025
            exposure_qc.ObjectiveStatusDate.cast("date").alias("ObjectiveStatusDate"),
            lit(None).cast("date").alias("GracePeriodEndDate"),
            lit(None).cast("string").alias("GracePeriodTypeCode"),
            lit(None).cast("string").alias("GracePeriodType"),
            lit(None).cast("string").alias("GracePeriodInterestPayableFlag"),
            lit(None).cast("timestamp").alias("GracePeriodNextInterestDate"),
            lit(None).cast("decimal(3,0)").alias("GracePeriodInterestFrequency"),
            exposure_qc.off_balance.cast("decimal(19,2)").alias("OffBalance"),
            exposure_qc.on_balance.cast("decimal(19,2)").alias("OnBalance"),
            exposure_qc.total_balance.cast("string").alias("OutstandingTotalBalance"),
            exposure_qc.principal.cast("decimal(19,2)").alias("Principal"),
            exposure_qc.interest_amount.cast("decimal(19,2)").alias("Interest"),
            lit(None).cast("decimal(19,2)").alias("UnrealizedInterest"),
            lit(None).cast("decimal(19,2)").alias("Commission"),
            exposure_qc.expenses.cast("decimal(19,2)").alias("Expense"),
            lit(None).cast("decimal(19,2)").alias("Accrued"),
            exposure_qc.overdue_other_amount.cast("decimal(26,4)").alias("DueAmount"),
            exposure_qc.overdue_principal.cast("decimal(19,2)").alias("DuePrincipal"),
            lit(None).cast("decimal(19,2)").alias("DueInterest"),
            lit(None).cast("decimal(19,2)").alias("DueUnrealizedInterest"),
            lit(None).cast("decimal(19,2)").alias("DueCommission"),
            lit(None).cast("decimal(19,2)").alias("DueExpense"),
            lit(None).cast("decimal(19,2)").alias("DueAccrued"),
            lit(None).cast("decimal(19,2)").alias("PenaltyInterest"),
            lit(None).cast("decimal(19,2)").alias("PenaltyUnrealizedInterest"),
            lit(None).cast("decimal(19,6)").alias("InterestRate"),
            lit(None).cast("int").alias("InterestCode"),
            lit(None).alias("InterestDescription"),
            lit(None).cast("decimal(19,6)").alias("BaseRate"),
            lit(None).alias("BaseCode"),
            lit(None).alias("BaseDescription"),
            lit(None).alias("N128Rate"),
            lit(None).cast("decimal(19,6)").alias("SpreadRate"),
            lit(None).cast("decimal(19,6)").alias("PenaltyRate"),
            lit(None).alias("InterestRateTypeCode"),
            lit(None).alias("InterestRateType"),
            lit(None).alias("SpreadCurrencyCode"),
            lit(None).alias("SpreadCurrency"),
            lit(None).alias("SpreadPeriod"),
            lit(None).alias("InterestDuration"),
            lit(None).alias("DayCounts"),
            lit(None).alias("InterestFrequency"),
            lit(None).alias("InterestFixedDate"),
            lit(None).alias("InterestValidityDate"),
            lit(None).alias("InterestChangeStartDate"),
            lit(None).alias("InterestChangeEndDate"),
            lit(None).alias("InterestRateUpdateDate"),
            lit(None).alias("InterestCalcStopFlag"),
            lit(None).alias("AutoRenewalFlag"),
            exposure_qc.case_currently_open_flag.cast("boolean").alias("IsActiveFlag"),
            lit(None).alias("WriteOffAmount"),
            exposure_qc.WriteOff_Date.alias("WriteOffDate"),
            lit(None).cast("string").alias("AssetSegmentCode"),
            lit(None).cast("string").alias("AssetSegment"),
            case_master.LPS.cast("string").alias("LPSCode"),
            case_master.lps_status.cast("string").alias("LPS"),
            lps_priority.LPS.cast("string").alias("LPSEng"),
            case_master.lps_date.cast("string").alias("LPSDate"),
            lit(None).cast("string").alias("DiasNumber"),
            lit(None).cast("string").alias("ProvisionFrozenInterestAmount"),
            lit(None).cast("string").alias("AccountNticeBUKey"),
            lit(None).cast("string").alias("SayFlag"),
            lit(None).cast("string").alias("InsuranceCompany"),
            lit(None).cast("string").alias("PhysicalNumber"),
            lit(None).cast("string").alias("PreCourtDate"),
            lit(None).cast("string").alias("CourtDate"),
            lit(None).cast("string").alias("ContractSigningDate"),
            lit(None).alias("ExposureLimitAmount"),
            lit(None).alias("BaloonAmount"),
            debt.DEBT_ADMINISTRATION_STATUS_CODE.cast("int").alias(
                "AdministrationStatusCode"
            ),
            debt.DEBT_ADMINISTRATION_STATUS_CODE.cast("int").alias(
                "AdministrationStatus"
            ),
            lit(None).alias("Tenor"),
            lit(None).alias("IsSoldFlag"),
            lit(None).alias("IsSubsidyFlag"),
            lit(None).alias("IsGuaranteeFlag"),
            lit(None).alias("EverDenouncedFlag"),
            lit(None).alias("EverDenouncedDate"),
            lit(None).alias("SPVCode"),
            lit(None).alias("SPV"),
            lit(None).alias("TotalOverdueInsterest"),
            lit(None).alias("TotalOverdueInstallments"),
            lit(None).alias("OutstandingTotalBalanceMLT"),
            case_master.CEXT_GENERIC_BIT_03.alias("AllBorrowersDeceasedFlag"),
            lit(None).alias("TreatmentBranchCode"),
            lit(None).alias("TreatmentBranch"),
            case_master.casp_generic_int_09_code.cast("string").alias(
                "LetterStatusCode"
            ),
            case_master.casp_generic_int_09_code.cast("string").alias("LetterStatus"),
            lit(None).alias("LetterStatusDate"),
            exposure_qc.CoC_AccountStatus.cast("string").alias("CoCStatusCode"),
            exposure_qc.CoC_AccountStatus.cast("string").alias("CoCStatus"),
            exposure_qc.CoC_AccountStatusDate.cast("date").alias(
                "CoCAccountStatusDate"
            ),
            exposure_qc.First_CoC_LetterDate.cast("date").alias("FirstLetterDate"),
            exposure_qc.Second_CoC_LetterDate.cast("date").alias("SecondLetterDate"),
            exposure_qc.Third_CoC_LetterDate.cast("date").alias("ThirdLetterDate"),
            exposure_qc.CEXT_GENERIC_DATETIME_08.cast("date").alias(
                "FirstLetterReminderDate"
            ),
            exposure_qc.FirstLetterFlag.cast("boolean").alias("FirstLetterFlag"),
            exposure_qc.FirstLetterReminderFlag.cast("boolean").alias(
                "FirstLetterReminderFlag"
            ),
            exposure_qc.SecondLetterFlag.cast("boolean").alias("SecondLetterFlag"),
            exposure_qc.ThirdLetterFlag.cast("boolean").alias("ThirdLetterFlag"),
            exposure_qc.ExclusionDate.cast("date").alias("ExclusionDate"),
            exposure_qc.ExclusionReasonCode.cast("string").alias("ExclusionReasonCode"),
            exposure_qc.ExclusionReason.cast("string").alias("ExclusionReason"),
            lit(None).alias("LegalActionPriority"),
            lit(None).alias("DueDebtAmount"),
            lit(None).alias("JustificationCode"),
            lit(None).alias("JustificationComments"),
            exposure_qc.businessdate.cast("date").alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"), 
            lit(None).cast("decimal(15,2)").alias("url_nrm_int_amn"), 
        )

        # ========================== Append ==========================
        exposure_qc.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_exposure_qc(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)